# Game resource library audit

## Findings at a glance

The library contains **3,922 canonical project locations** and **191 inspected element records**. Most locations came from discovery lists. The bulk asset tables add **153 glTF model descriptors** and **4,239 inspected SVG icons**. These are separate units of work.

All 4,239 icons have a license classification. **61 lack an individual credit**, which affects the reuse decision even though a license link exists. The worked examples below compare skeletal animation, keyboard-accessible controls and Blender material workflows at their respective evidence levels. [Source coverage](DISCOVERY.md) · [Model method](MODELS.md) · [Audit decisions](QUALITY.md)

## Context and methods

The decision is which resource a character artist or gameplay engineer can try, which requires a rights check, and which only merits a code study. The notebook reads a released SQLite snapshot in read-only mode. Run all cells from the library directory or repository root. Python's standard library and a Jupyter Python kernel suffice. SQLite must include FTS5 and JSON functions. The HTML tables and chart also include text fallbacks.

### Key assumptions

A project ID represents a canonical location and can have several source observations. An element represents a curated file or symbol at a revision. An asset represents a source path, including a verified SVG or a parsed glTF JSON descriptor. A fork row represents one upstream GitHub repository ID. Source tags describe a lead. An executed finding names the exact imported or run unit. Capturing a stable hash establishes identity but grants no usage rights.

In [1]:
from hashlib import sha256
from html import escape
from pathlib import Path
from time import perf_counter
import json
import sqlite3
import sys

started = perf_counter()
candidates = [Path("library.sqlite"), Path("research/game-resource-library/library.sqlite")]
database = next((path.resolve() for path in candidates if path.is_file()), None)
if database is None:
    raise FileNotFoundError("Open the notebook from the resource library or repository root.")
connection = sqlite3.connect(database.as_uri() + "?mode=ro", uri=True)
connection.row_factory = sqlite3.Row
connection.execute("PRAGMA query_only = ON")

def query(sql, parameters=()):
    return connection.execute(sql, parameters).fetchall()

def scalar(sql, parameters=()):
    return connection.execute(sql, parameters).fetchone()[0]

def file_digest(path):
    digest = sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

class Table:
    def __init__(self, rows):
        self.rows = rows

    def _repr_html_(self):
        if not self.rows:
            return "<p>No matches</p>"
        columns = self.rows[0].keys()
        header = "".join("<th scope='col'>" + escape(k.replace("_", " ")) + "</th>" for k in columns)
        body = "".join("<tr>" + "".join("<td>" + escape(str(row[k])) + "</td>" for k in columns) + "</tr>" for row in self.rows)
        return "<table><thead><tr>" + header + "</tr></thead><tbody>" + body + "</tbody></table>"

    def __repr__(self):
        if not self.rows:
            return "No matches"
        return "\n".join([" | ".join(self.rows[0].keys())] + [" | ".join(map(str, row)) for row in self.rows])

integrity = scalar("PRAGMA integrity_check")
assert integrity == "ok" and not query("PRAGMA foreign_key_check")
for source, index in [("elements","element_search"), ("projects","project_search"), ("assets","asset_search")]:
    for left, right in [(source, index), (index, source)]:
        assert not query(f"SELECT id FROM {left} EXCEPT SELECT id FROM {right} LIMIT 1")
    assert scalar(f"SELECT COUNT(*) FROM {source}") == scalar(f"SELECT COUNT(*) FROM {index}")

print(f"{database.name}, SHA-256 {file_digest(database)}")
print(f"Python {sys.version.split()[0]}, SQLite {sqlite3.sqlite_version}, integrity {integrity}. All three FTS ID sets reconcile.")

library.sqlite, SHA-256 ced483ca959419ea94f7f6cabf3ee34b941479dfca013f32795cd370a8e27672
Python 3.12.14, SQLite 3.53.1, integrity ok. All three FTS ID sets reconcile.


## Data and coverage

### Count each grain separately

The row excess over distinct IDs should be zero for the primary registries. Several observations can describe the same project. Multiple elements can describe different uses of the same bytes. Repeated display names can refer to different files or repositories.

In [2]:
Table(query("""
WITH registry(grain, records, distinct_key, key_unit) AS (
  SELECT 'Project locations', COUNT(*), COUNT(DISTINCT id), 'canonical project ID' FROM projects
  UNION ALL SELECT 'Catalog observations', COUNT(*), COUNT(DISTINCT project_id), 'project described' FROM project_sources
  UNION ALL SELECT 'Element IDs', COUNT(*), COUNT(DISTINCT id), 'curator element ID' FROM elements
  UNION ALL SELECT 'Inspected element content', COUNT(*), COUNT(DISTINCT sha256), 'SHA-256' FROM elements
  UNION ALL SELECT kind, COUNT(*), COUNT(DISTINCT sha256), 'SHA-256 of inspected file' FROM assets GROUP BY kind
  UNION ALL SELECT 'Direct REA forks', COUNT(*), COUNT(DISTINCT id), 'GitHub repository ID' FROM rea_forks
)
SELECT grain, records, distinct_key, key_unit, records - distinct_key AS excess_rows,
       ROUND(100.0 * (records - distinct_key) / records, 3) AS excess_percent
FROM registry
"""))

grain,records,distinct key,key unit,excess rows,excess percent
Project locations,3922,3922,canonical project ID,0,0.0
Catalog observations,4544,3922,project described,622,13.688
Element IDs,191,191,curator element ID,0,0.0
Inspected element content,191,190,SHA-256,1,0.524
3D model,153,152,SHA-256 of inspected file,1,0.654
Icon,4239,4239,SHA-256 of inspected file,0,0.0
Direct REA forks,116393,116393,GitHub repository ID,0,0.0


### Find overlapping discovery sources

Pairs below count distinct shared project IDs. A location can appear in several pairs. The overlap total uses six discovery families and excludes the synthetic “Inspected source repository” label added by the library builder. [Discovery inputs and reconciliation](data/discovery-counts.json)

In [3]:
families = """
SELECT DISTINCT project_id, name
FROM project_sources WHERE name <> 'Inspected source repository'
"""
observations = scalar("SELECT COUNT(*) FROM project_sources WHERE name <> 'Inspected source repository'")
unique_projects = scalar(f"SELECT COUNT(DISTINCT project_id) FROM ({families})")
overlap = scalar(f"SELECT COUNT(*) FROM (SELECT project_id FROM ({families}) GROUP BY project_id HAVING COUNT(*) > 1)")
print(f"{observations:,} recorded discovery observations → {unique_projects:,} locations. {overlap:,} locations appear in multiple discovery families ({overlap/unique_projects:.1%}).")
Table(query(f"""
WITH families AS ({families})
SELECT a.name AS first_source, b.name AS second_source, COUNT(*) AS shared_projects
FROM families a JOIN families b ON b.project_id=a.project_id AND a.name < b.name
GROUP BY a.name, b.name ORDER BY shared_projects DESC, a.name, b.name LIMIT 6
"""))

4,492 recorded discovery observations → 3,870 locations. 578 locations appear in multiple discovery families (14.9%).


first source,second source,shared projects
Engine reimplementations,Open Source Game Clones,506
Awesome REA Games,Open Source Game Clones,43
Engine reimplementations,GitHub topic search,16
GitHub topic search,Inspected code modules,15
Awesome REA Games,Engine reimplementations,11
Engine reimplementations,Inspected code modules,8


### Count language labels and separate unknowns

The multi-value language list is expanded per project. A project can count in more than one language, so the column does not sum to total projects. The separate fork table records GitHub’s single reported language, which is often missing.

In [4]:
unknown_project = scalar("SELECT COUNT(*) FROM projects WHERE COALESCE(json_array_length(metadata,'$.languages'),0)=0")
unknown_fork = scalar("SELECT COUNT(*) FROM rea_forks WHERE NULLIF(trim(github_language),'') IS NULL")
forks = scalar("SELECT COUNT(*) FROM rea_forks")
print(f"{unknown_project:,} project locations have no multi-value language entries. {unknown_fork:,} of {forks:,} direct forks lack a reported language ({unknown_fork/forks:.1%}).")
Table(query("""
SELECT language.value AS listed_language, COUNT(DISTINCT p.id) AS project_locations
FROM projects p, json_each(p.metadata,'$.languages') language
WHERE NULLIF(trim(language.value),'') IS NOT NULL
GROUP BY lower(language.value) ORDER BY project_locations DESC, listed_language LIMIT 9
"""))

172 project locations have no multi-value language entries. 106,949 of 116,393 direct forks lack a reported language (91.9%).


listed language,project locations
C++,1072
C,912
Assembly,525
Rust,235
JavaScript,200
C#,196
Python,174
Java,149
Lua,92


## Results

### Separate inspection from discovery and weigh the reuse decision

The chart counts curated elements. A source read, format inspection or observed execution records different evidence. The handling decision also considers the file’s rights, dependencies and project fit. An executed probe can still land in legal review.

In [5]:
outcomes = query("""
SELECT evidence, "use" AS decision, COUNT(*) AS elements
FROM elements GROUP BY evidence, "use" ORDER BY evidence, decision
""")
class DecisionChart:
    def _repr_html_(self):
        levels = ["source-read", "format-inspected", "executed"]
        choices = ["adapt", "study", "verify-terms"]
        counts = {(row["evidence"], row["decision"]): row["elements"] for row in outcomes}
        colors = ["#126450", "#345984", "#955223"]
        peak = max(sum(counts.get((level, choice), 0) for choice in choices) for level in levels)
        parts = ["<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 620 185' width='620' style='max-width:100%;height:auto' role='img' aria-labelledby='ct'><title id='ct'>Inspected elements by evidence and handling decision, counts per row</title><rect width='620' height='185' fill='#fff'/>"]
        for i, level in enumerate(levels):
            x, y = 138, 14 + i*39
            total = sum(counts.get((level, choice), 0) for choice in choices)
            parts.append(f"<text x='6' y='{y+18}' font-size='15' fill='#222'>{level}</text>")
            for choice, color in zip(choices, colors):
                number = counts.get((level, choice), 0)
                width = 410 * number / peak
                parts.append(f"<rect x='{x:.2f}' y='{y}' width='{width:.2f}' height='24' fill='{color}'><title>{level}, {choice}, {number} elements</title></rect>")
                x += width
            parts.append(f"<text x='{x+7:.2f}' y='{y+18}' font-size='15' fill='#222'>{total}</text>")
        for i, (choice,color) in enumerate(zip(choices,colors)):
            x = 9+i*180
            parts.append(f"<rect x='{x}' y='146' width='16' height='16' fill='{color}'/><text x='{x+22}' y='160' font-size='15' fill='#222'>{choice}</text>")
        return "".join(parts) + "</svg>" + Table(outcomes)._repr_html_()

    def __repr__(self):
        return repr(Table(outcomes))

DecisionChart()

evidence,decision,elements
executed,adapt,12
executed,verify-terms,1
format-inspected,adapt,17
format-inspected,verify-terms,1
source-read,adapt,83
source-read,study,62
source-read,verify-terms,15


### Inspect duplicate bytes and license gaps

Identical model descriptors can have different sibling textures or buffers. A curated record may explain a distinct behavior in the same source file. A count of duplicate hashes is useful for caching work, while deleting those rows requires checking the surrounding evidence.

In [6]:
Table(query("""
WITH files AS (
  SELECT 'curated element' AS collection, id, sha256 FROM elements
  UNION ALL SELECT kind, id, sha256 FROM assets
)
SELECT collection, COUNT(*) AS records, substr(sha256,1,12) AS digest_prefix,
       group_concat(id, ', ') AS ids
FROM files GROUP BY collection, sha256 HAVING COUNT(*)>1
ORDER BY records DESC, collection, sha256 LIMIT 8
"""))

collection,records,digest prefix,ids
3D model,2,1e9003a4a2a8,"khronos-gltf:BoxTextured, khronos-gltf:BoxTexturedNonPowerOfTwo"
curated element,2,60b2e76bbc6c,"openra-radar, ui-openra-visibility-aware-radar-layers"


In [7]:
Table(query("""
SELECT kind, COUNT(*) AS files,
       SUM(NULLIF(trim(license),'') IS NULL) AS no_license,
       SUM(NULLIF(trim(license_url),'') IS NULL) AS no_license_link,
       SUM(NULLIF(trim(author),'') IS NULL) AS no_individual_credit,
       ROUND(100.0 * SUM(NULLIF(trim(author),'') IS NULL) / COUNT(*),2) AS missing_credit_percent
FROM assets GROUP BY kind
"""))

kind,files,no license,no license link,no individual credit,missing credit percent
3D model,153,0,0,0,0.0
Icon,4239,0,0,61,1.44


In [8]:
missing = query("""
SELECT COALESCE(NULLIF(trim(license),''),'unknown') AS classification,
       COUNT(*) AS icons_without_individual_credit
FROM assets WHERE kind='Icon' AND NULLIF(trim(author),'') IS NULL GROUP BY classification
""")
print(f"Curated elements without a terms link: {scalar('SELECT COUNT(*) FROM elements WHERE license_url IS NULL')}.")
assert not query("""SELECT id FROM elements WHERE "use"='adapt' AND license_url IS NULL""")
Table(missing)

Curated elements without a terms link: 5.


classification,icons without individual credit
CC-BY-3.0,61


### Find a rig with actual animation

A skinned, animated glTF candidate must declare both a skin and an animation. The model table records parsed JSON. A separate element can establish that a particular GLB variant was imported into Blender. The join uses the pinned project, revision and model directory and counts executed GLB imports only. The descriptor and its binary variant have different content hashes.

In [9]:
rig_candidates = query("""
SELECT a.name, json_extract(a.metadata,'$.counts.joints') AS joints,
       json_extract(a.metadata,'$.counts.animations') AS clips, a.license,
       EXISTS (
         SELECT 1 FROM elements e WHERE e.project_id=a.project_id AND e.revision=a.revision
         AND e.path LIKE 'Models/%/%.glb' AND e.evidence='executed'
         AND a.path LIKE 'Models/' || substr(e.path,8,instr(substr(e.path,8),'/')-1) || '/%'
       ) AS recorded_glb_execution
FROM assets a WHERE a.kind='3D model' AND json_extract(a.metadata,'$.counts.skins')>0
AND json_extract(a.metadata,'$.counts.animations')>0 ORDER BY recorded_glb_execution DESC, a.name
""")
Table(rig_candidates)

name,joints,clips,license,recorded glb execution
Cesium Man,19,1,CC-BY-4.0 + LicenseRef-LegalMark-Cesium,1
Fox,24,3,CC0-1.0 + CC-BY-4.0,1
Rigged Figure,19,1,CC-BY-4.0,1
BrainStem,18,1,LicenseRef-Poser-EULA,0
Recursive Skeletons,840,1,CC-BY-4.0,0
Rigged Simple,2,1,CC-BY-4.0,0
ScatterAnemone,33,1,CC-BY-4.0,0
Simple Skin,2,1,CC0-1.0,0


### Find a reusable keyboard button or material workflow

Here the query searches the inspected element content and then filters by domain. It shows both adaptable and study-only results. To inspect a source, use the stable element ID in the offline explorer and follow its revision and license links. A keyboard capture button and a modal focus manager address different interaction needs.

In [10]:
def find_elements(expression, categories, limit):
    slots = ",".join("?" for _ in categories)
    return query(f"""
      SELECT e.id, e.name, e.evidence, e."use" AS decision, e.license
      FROM element_search f JOIN elements e ON e.id=f.id
      WHERE element_search MATCH ? AND e.category IN ({slots})
      ORDER BY bm25(element_search), e.id LIMIT ?
    """, (expression, *categories, limit))

Table(find_elements("button AND (focus OR keyboard)", ["ui"], 6))

id,name,evidence,decision,license
ui-godot-demo-projects-keyboard-remap-capture-button,Keyboard remap capture button,source-read,adapt,MIT
ui-libgdx-modal-focus-acquisition-and-restoration,Modal focus acquisition and restoration,source-read,adapt,Apache-2.0
ui-bevy-directional-focus-and-synthetic-activation,Directional focus and synthetic activation,source-read,adapt,MIT
ui-kenney-rpg-parchment-normal,Kenney RPG parchment button • Normal,format-inspected,adapt,CC0-1.0
ui-kenney-rpg-parchment-pressed,Kenney RPG parchment button • Pressed,format-inspected,adapt,CC0-1.0
ui-kenney-base-primary-disabled,Kenney blue button • Disabled,format-inspected,adapt,CC0-1.0


In [11]:
Table(find_elements("(material OR texture) AND (Blender OR glTF)", ["blender","graphics","build"], 6))

id,name,evidence,decision,license
art-helmet,Flight Helmet leather material channel pack,format-inspected,adapt,CC0-1.0
art-metalrough,Principled-to-glTF PBR material collector,source-read,adapt,Apache-2.0
art-texcompress,glTF Transform texture conversion,source-read,adapt,MIT
art-poly-disp,Poly Haven material displacement setup,source-read,adapt,GPL-3.0
art-principled,Principled BSDF shader sockets,source-read,adapt,GPL-2.0-or-later
art-atlasuv,MCprep UV and transparent-face tools,source-read,adapt,GPL-2.0-or-later


## Takeaways

Start rig prototyping with a model whose declared skin, clip inventory, Blender import and file-level rights fit the task. Keep cursor focus and keyboard remapping behavior separate when borrowing a UI control. For shaders, compare the export routine with an inspected channel-packed texture, then test the authored material under the receiving renderer.

Use source observations and fork lists to discover projects. Build estimates need a pinned source file, its dependencies and an execution fixture. The bulk sample corpus concentrates assets in two upstream repositories. A cross-sectional snapshot cannot establish project health, long-term maintenance or the total number of reusable games. [Research inputs](sources/) · [Bulk asset evidence](evidence/)

In [12]:
metrics = {
    "projects": scalar("SELECT COUNT(*) FROM projects"),
    "inspected_projects": scalar("SELECT COUNT(DISTINCT project_id) FROM elements"),
    "elements": scalar("SELECT COUNT(*) FROM elements"),
    "model_descriptors": scalar("SELECT COUNT(*) FROM assets WHERE kind='3D model'"),
    "inspected_svg": scalar("SELECT COUNT(*) FROM assets WHERE kind='Icon'"),
    "skinned_animated_descriptors": len(rig_candidates),
}
receipts = json.loads((database.parent / "evidence/quality.json").read_text())
assert metrics["model_descriptors"] == receipts["asset_kinds"]["3D model"]
assert metrics["inspected_svg"] == receipts["asset_kinds"]["Icon"]
assert metrics["elements"] == receipts["elements"] and metrics["projects"] == receipts["projects"]
assert file_digest(database) == receipts["sqlite_sha256"]
print(json.dumps(metrics, indent=2))
print(f"Recorded runtime: {perf_counter() - started:.3f}s. Source digest and count receipts reconciled.")
connection.close()

{
  "projects": 3922,
  "inspected_projects": 76,
  "elements": 191,
  "model_descriptors": 153,
  "inspected_svg": 4239,
  "skinned_animated_descriptors": 8
}
Recorded runtime: 0.543s. Source digest and count receipts reconciled.
